# Phase 4 — Defenses
Objectif : rendre le modele robuste contre FGSM et PGD.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import random
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('libs OK')

## 1. Recharger donnees + modele baseline

In [ ]:
def preprocess(train_df, test_df):
    drop_cols = ['id', 'attack_cat']
    train_df = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
    test_df  = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
    for df in [train_df, test_df]:
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
    median_vals = train_df.median(numeric_only=True)
    train_df.fillna(median_vals, inplace=True)
    test_df.fillna(median_vals, inplace=True)
    cat_cols = [c for c in train_df.select_dtypes(include=['object','str']).columns if c != 'label']
    le = LabelEncoder()
    for col in cat_cols:
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        known = set(le.classes_)
        test_df[col] = test_df[col].astype(str).apply(lambda x: x if x in known else le.classes_[0])
        test_df[col] = le.transform(test_df[col])
    X_train = train_df.drop('label', axis=1).values.astype(np.float32)
    y_train = train_df['label'].values.astype(np.float32)
    X_test  = test_df.drop('label', axis=1).values.astype(np.float32)
    y_test  = test_df['label'].values.astype(np.float32)
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)
    return X_train, y_train, X_test, y_test, scaler

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1),          nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

train_df = pd.read_csv('../data/UNSW_NB15_training-set.csv')
test_df  = pd.read_csv('../data/UNSW_NB15_testing-set.csv')
X_train, y_train, X_test, y_test, scaler = preprocess(train_df.copy(), test_df.copy())

baseline = MLP(input_dim=X_train.shape[1])
baseline.load_state_dict(torch.load('../results/baseline_model.pth', weights_only=True))
baseline.eval()

cols = pd.read_csv('../data/UNSW_NB15_training-set.csv').drop(columns=['id','attack_cat','label']).columns.tolist()
MANIPULABLE = ['dur','spkts','dpkts','sbytes','dbytes','sttl','sload','dload','sinpkt','dinpkt','sjit','djit','smean','dmean']
manipulable_idx = [cols.index(f) for f in MANIPULABLE if f in cols]
mask = torch.zeros(len(cols))
mask[manipulable_idx] = 1.0
print('Tout charge OK')

## 2. Fonctions attaques + evaluation (rappel)

In [ ]:
def fgsm_attack(model, X, y, epsilon, mask):
    X_t = torch.FloatTensor(X).requires_grad_(True)
    loss = nn.BCELoss()(model(X_t).squeeze(), torch.FloatTensor(y))
    loss.backward()
    return (X_t + epsilon * X_t.grad.sign() * mask).detach().numpy()

def pgd_attack(model, X, y, epsilon, alpha, steps, mask):
    X_orig = torch.FloatTensor(X)
    X_adv  = X_orig.clone()
    y_t    = torch.FloatTensor(y)
    for _ in range(steps):
        X_adv = X_adv.detach().requires_grad_(True)
        loss  = nn.BCELoss()(model(X_adv).squeeze(), y_t)
        loss.backward()
        X_adv = X_orig + torch.clamp(X_adv + alpha * X_adv.grad.sign() * mask - X_orig, -epsilon, epsilon)
    return X_adv.detach().numpy()

def evaluate(model, X, y_true, label=''):
    model.eval()
    with torch.no_grad():
        prob = model(torch.FloatTensor(X)).squeeze().numpy()
        pred = (prob >= 0.5).astype(int)
    attack_mask = (y_true == 1)
    evasion = (pred[attack_mask] == 0).mean() * 100
    r = {'F1': f1_score(y_true, pred), 'Precision': precision_score(y_true, pred),
         'Recall': recall_score(y_true, pred), 'PR-AUC': average_precision_score(y_true, prob),
         'Evasion%': evasion}
    if label:
        print(f'--- {label} ---')
        for k, v in r.items(): print(f'  {k:12s}: {v:.4f}')
    return r

attack_idx = np.where(y_test == 1)[0]
X_attacks  = X_test[attack_idx]
y_attacks  = y_test[attack_idx]

EPS = 0.1
X_fgsm_full = X_test.copy()
X_fgsm_full[attack_idx] = fgsm_attack(baseline, X_attacks, y_attacks, EPS, mask)
X_pgd_full  = X_test.copy()
X_pgd_full[attack_idx]  = pgd_attack(baseline, X_attacks, y_attacks, EPS, 0.01, 40, mask)

print('=== Baseline avant defense ===')
r_clean = evaluate(baseline, X_test,       y_test, 'Donnees propres')
r_fgsm  = evaluate(baseline, X_fgsm_full,  y_test, f'Sous FGSM eps={EPS}')
r_pgd   = evaluate(baseline, X_pgd_full,   y_test, f'Sous PGD  eps={EPS}')

## 3. Defense 1 — Adversarial Training
On entraine le modele sur un mix de donnees propres + exemples adversariaux.

In [ ]:
def adversarial_training(X_train, y_train, input_dim, epsilon=0.1, epochs=30, batch=512):
    model = MLP(input_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()
    losses = []

    for epoch in range(epochs):
        model.train()
        # Generer les exemples adversariaux sur le batch courant
        X_adv = fgsm_attack(model, X_train, y_train, epsilon, mask)
        # Mix 50/50 propres + adversariaux
        X_mix = np.concatenate([X_train, X_adv])
        y_mix = np.concatenate([y_train, y_train])
        idx   = np.random.permutation(len(X_mix))
        X_mix, y_mix = X_mix[idx], y_mix[idx]

        X_t = torch.FloatTensor(X_mix)
        y_t = torch.FloatTensor(y_mix)
        epoch_loss = 0
        for start in range(0, len(X_t), batch):
            xb = X_t[start:start+batch]
            yb = y_t[start:start+batch]
            optimizer.zero_grad()
            loss = criterion(model(xb).squeeze(), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg = epoch_loss / max(1, len(X_t) // batch)
        losses.append(avg)
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1:02d}/{epochs} | Loss: {avg:.4f}')

    return model, losses

print('Entrainement Adversarial Training...')
model_adv, losses_adv = adversarial_training(X_train, y_train, X_train.shape[1], epsilon=0.1)
torch.save(model_adv.state_dict(), '../results/adversarial_model.pth')
print('Sauvegarde : results/adversarial_model.pth')

## 4. Defense 2 — Feature Squeezing
On reduit la precision des features pour effacer les petites perturbations.

In [ ]:
def feature_squeezing(X, n_bits=4):
    max_val = np.max(np.abs(X), axis=0, keepdims=True) + 1e-8
    X_norm  = X / max_val
    levels  = 2 ** n_bits
    return np.round(X_norm * levels) / levels * max_val

X_test_sq      = feature_squeezing(X_test)
X_fgsm_sq_full = feature_squeezing(X_fgsm_full)
X_pgd_sq_full  = feature_squeezing(X_pgd_full)
print('Feature squeezing applique OK')

## 5. Evaluation comparative des 3 modeles

In [ ]:
model_adv.eval()

print('=== RESULTATS COMPLETS ===')
results = {}

# Baseline
results['Baseline / propres'] = evaluate(baseline,   X_test,       y_test)
results['Baseline / FGSM']    = evaluate(baseline,   X_fgsm_full,  y_test)
results['Baseline / PGD']     = evaluate(baseline,   X_pgd_full,   y_test)

# Adversarial Training
X_fgsm_adv = X_test.copy()
X_fgsm_adv[attack_idx] = fgsm_attack(model_adv, X_attacks, y_attacks, EPS, mask)
X_pgd_adv  = X_test.copy()
X_pgd_adv[attack_idx]  = pgd_attack(model_adv, X_attacks, y_attacks, EPS, 0.01, 40, mask)

results['Adv.Training / propres'] = evaluate(model_adv, X_test,      y_test)
results['Adv.Training / FGSM']    = evaluate(model_adv, X_fgsm_adv,  y_test)
results['Adv.Training / PGD']     = evaluate(model_adv, X_pgd_adv,   y_test)

# Feature Squeezing (utilise le baseline)
results['Feat.Squeeze / propres'] = evaluate(baseline, X_test_sq,      y_test)
results['Feat.Squeeze / FGSM']    = evaluate(baseline, X_fgsm_sq_full, y_test)
results['Feat.Squeeze / PGD']     = evaluate(baseline, X_pgd_sq_full,  y_test)

print('\n{:<30s} {:>8s} {:>8s} {:>10s}'.format('Scenario', 'F1', 'PR-AUC', 'Evasion%'))
print('-' * 60)
for k, v in results.items():
    print('{:<30s} {:>8.4f} {:>8.4f} {:>9.1f}%'.format(k, v['F1'], v['PR-AUC'], v['Evasion%']))

## 6. Graphique comparatif final

In [ ]:
scenarios   = ['Propres', 'FGSM', 'PGD']
f1_baseline = [results['Baseline / propres']['F1'],   results['Baseline / FGSM']['F1'],   results['Baseline / PGD']['F1']]
f1_adv      = [results['Adv.Training / propres']['F1'],results['Adv.Training / FGSM']['F1'],results['Adv.Training / PGD']['F1']]
f1_sq       = [results['Feat.Squeeze / propres']['F1'],results['Feat.Squeeze / FGSM']['F1'],results['Feat.Squeeze / PGD']['F1']]

x     = np.arange(len(scenarios))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, f1_baseline, width, label='Baseline',          color='steelblue')
ax.bar(x,         f1_adv,      width, label='Adv. Training',     color='tomato')
ax.bar(x + width, f1_sq,       width, label='Feature Squeezing', color='seagreen')
ax.set_ylabel('F1-score')
ax.set_title('Comparaison de robustesse : Baseline vs Defenses')
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.legend()
ax.set_ylim(0, 1.05)
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=8)
plt.tight_layout()
plt.savefig('../results/defense_comparison.png', dpi=150)
plt.show()
print('Sauvegarde : results/defense_comparison.png')
print('=> Phase 5 : resultats finaux + rapport')